# OMGM Step 2: Fine-Grained Multimodal (IT2IT) Reranking on Kaggle

This notebook executes **Step 2** of the OMGM pipeline on the 1,000 InfoSeek test queries:
1. Installs required dependencies (including compatible `transformers<4.41.0`).
2. Clones the official OMGM repository to access the specialized `lavis` library and `qformer_IT2IT_reranker`.
3. Loads coarse retrieval candidates from Step 1 (`result.json`, top-20 entities per query).
4. Instantly loads candidate Wikipedia image mappings from `candidate_wiki_images_max3.csv`.
5. Loads the trained Q-Former IT2IT reranker checkpoint (`model_23912.pth`).
6. Extracts fine-grained cross-modal token embeddings between query (image + question) and candidate pairs (entity image + article section text).
7. Blends Step 1 coarse similarities with Step 2 fine-grained multimodal similarities: $\text{score} = 0.9 \times s_1 + 0.1 \times s_2$.
8. Evaluates final **Recall@1, @5, @10, @20** and exports `/kaggle/working/step2_results/result.json`.

### Cell 1: Environment Setup & Clone OMGM Repository
Installs dependencies with compatible transformers (`<4.41.0`) and clones the OMGM repository.

In [ ]:
# 1. Install all required dependencies (using Kaggle default transformers)
!pip install -q timm einops fairscale faiss-cpu iopath decord webdataset omegaconf pycocoevalcap

import os, sys

# 2. Clone official OMGM repository if not already present
REPO_DIR = "/kaggle/working/OMGM"
if not os.path.exists(REPO_DIR):
    print("Cloning OMGM repository...")
    !git clone https://github.com/ChaoLinAViy/OMGM.git /kaggle/working/OMGM
else:
    print("OMGM repository already present.")

# Add OMGM root to python path
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)
print(f"Current working directory set to: {os.getcwd()}")


### Cell 2: Dataset Path Configuration & Verification
Verifies that all 7 dataset components exist at the Kaggle input paths.

In [ ]:
import os, glob, json, csv, time
import torch
import PIL.Image
from tqdm import tqdm

print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device:         {torch.cuda.get_device_name(0)}")
device = "cuda" if torch.cuda.is_available() else "cpu"

BASE_DS = "/kaggle/input/datasets/alphacentauriv/agentic-vqa-ds"

# 1. 1,000 test query samples
SAMPLE_FILE = os.path.join(BASE_DS, "infoseek_test_sample_1000.csv")

# 2. Step 1 results
STEP1_RESULT = os.path.join(BASE_DS, "result.json")

# 3. Trained Reranker Checkpoint
RERANKER_CKPT = os.path.join(BASE_DS, "reranker_model.pth/reranker_model.pth/model_23912.pth")

# 4. 100k Wikipedia Knowledge Base JSON
KB_PATH = os.path.join(BASE_DS, "infoseek_100k_wiki/wiki_100_dict_v4.json")

# 5. Sanitized Candidate Wikipedia Images Metadata CSV
CANDIDATE_CSV = os.path.join(BASE_DS, "candidate_wiki_images_max3.csv")

# 6. Candidate Wikipedia Images root directory
WIKI_IMG_PREFIX = os.path.join(BASE_DS, "wiki_imgs/wiki_imgs")

# 7. Query Images folder (oven_*.jpg)
QUERY_IMG_DIR = os.path.join(BASE_DS, "infoseek_val_1000_images/infoseek_val_1000_images")
if not os.path.isdir(QUERY_IMG_DIR):
    QUERY_IMG_DIR = os.path.join(BASE_DS, "infoseek_val_1000_images")

# 8. Output save directory
OUTPUT_DIR = "/kaggle/working/step2_results"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Verify paths
print("\n--- Verifying Paths ---")
for name, p in [
    ("Sample CSV", SAMPLE_FILE),
    ("Step 1 Result", STEP1_RESULT),
    ("Reranker Checkpoint", RERANKER_CKPT),
    ("Knowledge Base", KB_PATH),
    ("Candidate CSV", CANDIDATE_CSV),
    ("Wiki Images Prefix", WIKI_IMG_PREFIX),
    ("Query Images Dir", QUERY_IMG_DIR),
]:
    exists = os.path.exists(p)
    print(f"  {name:<20}: {'[OK]' if exists else '[MISSING]'} {p}")

### Cell 3: Fast Candidate Image Loader & Helper Functions
Loads candidate images directly from `candidate_wiki_images_max3.csv` in < 1 second.

In [ ]:
from utils import load_csv_data, get_test_question
from data_utils import targetpad_transform, process_images_in_parallel
from model import WikipediaKnowledgeBase, reconstruct_wiki_sections

def eval_recall(candidates, ground_truth, top_ks=[1, 5, 10, 20]):
    recall = {k: 0 for k in top_ks}
    for k in top_ks:
        if ground_truth in candidates[:k]:
            recall[k] = 1
    return recall

def load_candidate_wiki_images(csv_path, wiki_img_prefix):
    title2wikiimg = {}
    found_count = 0
    missing_count = 0
    candidate_prefixes = [
        wiki_img_prefix,
        os.path.join(wiki_img_prefix, "wiki_imgs"),
        os.path.join(wiki_img_prefix, "full"),
    ]
    with open(csv_path, "r", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for row in reader:
            title = row.get("title", "").strip()
            rel_path = row.get("relative_path", "").strip()
            caption = row.get("caption", "").strip()
            if not title or not rel_path:
                continue
            img_path = None
            for pfx in candidate_prefixes:
                test_p = os.path.join(pfx, rel_path)
                if os.path.exists(test_p):
                    img_path = test_p
                    break
            if img_path is None:
                img_path = os.path.join(wiki_img_prefix, rel_path)
                missing_count += 1
            else:
                found_count += 1
            if title not in title2wikiimg:
                title2wikiimg[title] = []
            if len(title2wikiimg[title]) < 10:
                cur_captions = [x["caption"] for x in title2wikiimg[title]]
                if caption not in cur_captions:
                    title2wikiimg[title].append({"caption": caption, "img_path": img_path})
    print(f"Loaded {len(title2wikiimg)} entity image records from candidate CSV.")
    print(f"  Images on disk : {found_count}")
    print(f"  Not on disk    : {missing_count}")
    return title2wikiimg

def resolve_query_image(image_id, query_img_dir):
    clean_id = image_id.split("|")[0].strip()
    for ext in ["", ".jpg", ".jpeg", ".JPEG", ".png"]:
        candidate = os.path.join(query_img_dir, clean_id + ext)
        if os.path.exists(candidate):
            return candidate
    matches = glob.glob(os.path.join(query_img_dir, f"*{clean_id}*"))
    if matches:
        return matches[0]
    raise FileNotFoundError(f"Query image {image_id} not found in {query_img_dir}")

### Cell 4: Load Knowledge Base & Q-Former IT2IT Reranker Checkpoint
Applies full Transformers compatibility bridge, loads Wikipedia articles into memory, and loads the pretrained Q-Former reranker weights.

In [ ]:
# ---------------------------------------------------------------------------
# HuggingFace Transformers Comprehensive Compatibility Bridge
# In newer transformers (>=4.45), apply_chunking_to_forward was moved,
# model_parallel_utils was removed, and pruning helpers were deprecated.
# We inject them into modeling_utils, pytorch_utils, and utils.model_parallel_utils
# so LAVIS's Qformer, med, and modeling_t5 import cleanly on modern Python.
# ---------------------------------------------------------------------------
import sys, types
import torch
import transformers.modeling_utils
import transformers.pytorch_utils

# 1. apply_chunking_to_forward
if hasattr(transformers.pytorch_utils, "apply_chunking_to_forward"):
    transformers.modeling_utils.apply_chunking_to_forward = transformers.pytorch_utils.apply_chunking_to_forward
elif hasattr(transformers.modeling_utils, "apply_chunking_to_forward"):
    transformers.pytorch_utils.apply_chunking_to_forward = transformers.modeling_utils.apply_chunking_to_forward

# 2. find_pruneable_heads_and_indices (both modules)
def find_pruneable_heads_and_indices(heads, n_heads, head_size, already_pruned_heads):
    mask = torch.ones(n_heads, head_size)
    heads = set(heads) - already_pruned_heads
    for head in heads:
        head = head - sum(1 if h < head else 0 for h in already_pruned_heads)
        mask[head] = 0
    mask = mask.view(-1).contiguous().eq(1)
    index = torch.arange(len(mask))[mask].long()
    return heads, index

transformers.modeling_utils.find_pruneable_heads_and_indices = find_pruneable_heads_and_indices
transformers.pytorch_utils.find_pruneable_heads_and_indices = find_pruneable_heads_and_indices

# 3. prune_linear_layer (both modules)
def prune_linear_layer(layer, index, dim=0):
    index = index.to(layer.weight.device)
    W = layer.weight.index_select(dim, index).clone().detach()
    if layer.bias is not None:
        b = layer.bias.clone().detach() if dim == 1 else layer.bias[index].clone().detach()
    new_size = list(layer.weight.size())
    new_size[dim] = len(index)
    new_layer = torch.nn.Linear(new_size[1], new_size[0], bias=layer.bias is not None).to(layer.weight.device)
    new_layer.weight.requires_grad = False
    new_layer.weight.copy_(W.contiguous())
    new_layer.weight.requires_grad = True
    if layer.bias is not None:
        new_layer.bias.requires_grad = False
        new_layer.bias.copy_(b.contiguous())
        new_layer.bias.requires_grad = True
    return new_layer

transformers.modeling_utils.prune_linear_layer = prune_linear_layer
transformers.pytorch_utils.prune_linear_layer = prune_linear_layer

# 4. model_parallel_utils (assert_device_map, get_device_map)
mpu = types.ModuleType("transformers.utils.model_parallel_utils")
mpu.assert_device_map = lambda *args, **kwargs: None
mpu.get_device_map = lambda *args, **kwargs: None
sys.modules["transformers.utils.model_parallel_utils"] = mpu
if hasattr(transformers, "utils"):
    transformers.utils.model_parallel_utils = mpu

# 5. file_utils (ModelOutput fallback)
if "transformers.file_utils" not in sys.modules:
    try:
        import transformers.file_utils
    except ModuleNotFoundError:
        fu = types.ModuleType("transformers.file_utils")
        from transformers.utils import ModelOutput
        fu.ModelOutput = ModelOutput
        sys.modules["transformers.file_utils"] = fu

# 6. all_tied_weights_keys & _tied_weights_keys (transformers >= 5.0 compatibility)
class TiedWeightsDescriptor:
    def __get__(self, obj, objtype=None):
        if obj is None:
            return {}
        if "_all_tied_weights_keys_storage" not in obj.__dict__:
            obj.__dict__["_all_tied_weights_keys_storage"] = {}
        return obj.__dict__["_all_tied_weights_keys_storage"]
    def __set__(self, obj, value):
        obj.__dict__["_all_tied_weights_keys_storage"] = value

if hasattr(transformers.modeling_utils, "PreTrainedModel"):
    if not hasattr(transformers.modeling_utils.PreTrainedModel, "all_tied_weights_keys"):
        transformers.modeling_utils.PreTrainedModel.all_tied_weights_keys = TiedWeightsDescriptor()
    if not hasattr(transformers.modeling_utils.PreTrainedModel, "_tied_weights_keys"):
        transformers.modeling_utils.PreTrainedModel._tied_weights_keys = []

# 7. Convert _keys_to_ignore_on_load_* from list to set (transformers >= 5.0 set operations)
if hasattr(transformers.modeling_utils.PreTrainedModel, "_adjust_missing_and_unexpected_keys"):
    _orig_adjust = transformers.modeling_utils.PreTrainedModel._adjust_missing_and_unexpected_keys
    def _patched_adjust(self, *args, **kwargs):
        if getattr(self, "_keys_to_ignore_on_load_unexpected", None) is not None:
            self._keys_to_ignore_on_load_unexpected = set(self._keys_to_ignore_on_load_unexpected)
            type(self)._keys_to_ignore_on_load_unexpected = set(self._keys_to_ignore_on_load_unexpected)
        if getattr(self, "_keys_to_ignore_on_load_missing", None) is not None:
            self._keys_to_ignore_on_load_missing = set(self._keys_to_ignore_on_load_missing)
            type(self)._keys_to_ignore_on_load_missing = set(self._keys_to_ignore_on_load_missing)
        return _orig_adjust(self, *args, **kwargs)
    transformers.modeling_utils.PreTrainedModel._adjust_missing_and_unexpected_keys = _patched_adjust
# 8. Synchronize cls.predictions.bias with resize_token_embeddings (30522 -> 30523)
if hasattr(transformers.modeling_utils.PreTrainedModel, "resize_token_embeddings"):
    _orig_resize_tokens = transformers.modeling_utils.PreTrainedModel.resize_token_embeddings
    def _patched_resize_tokens(self, new_num_tokens=None, pad_to_multiple_of=None):
        res = _orig_resize_tokens(self, new_num_tokens=new_num_tokens, pad_to_multiple_of=pad_to_multiple_of)
        if new_num_tokens is not None:
            if hasattr(self, "cls") and hasattr(self.cls, "predictions") and hasattr(self.cls.predictions, "bias"):
                pred = self.cls.predictions
                if pred.bias.shape[0] != new_num_tokens:
                    old_bias = pred.bias
                    new_bias = torch.nn.Parameter(torch.zeros(new_num_tokens, dtype=old_bias.dtype, device=old_bias.device))
                    n = min(old_bias.shape[0], new_num_tokens)
                    new_bias.data[:n] = old_bias.data[:n]
                    pred.bias = new_bias
                    if hasattr(pred, "decoder") and hasattr(pred.decoder, "bias"):
                        pred.decoder.bias = new_bias
        return res
    transformers.modeling_utils.PreTrainedModel.resize_token_embeddings = _patched_resize_tokens
# 9. get_head_mask compatibility (removed in newer transformers ModuleUtilsMixin)
def _convert_head_mask_to_5d(self, head_mask, num_hidden_layers):
    if head_mask.dim() == 1:
        head_mask = head_mask.unsqueeze(0).unsqueeze(0).unsqueeze(-1).unsqueeze(-1)
        head_mask = head_mask.expand(num_hidden_layers, -1, -1, -1, -1)
    elif head_mask.dim() == 2:
        head_mask = head_mask.unsqueeze(1).unsqueeze(-1).unsqueeze(-1)
    dtype = getattr(self, "dtype", torch.float32)
    return head_mask.to(dtype=dtype)

def get_head_mask(self, head_mask=None, num_hidden_layers=12, is_attention_chunked=False):
    if head_mask is not None:
        head_mask = self._convert_head_mask_to_5d(head_mask, num_hidden_layers)
        if is_attention_chunked:
            head_mask = head_mask.unsqueeze(-1)
    else:
        head_mask = [None] * num_hidden_layers
    return head_mask

transformers.modeling_utils.PreTrainedModel._convert_head_mask_to_5d = _convert_head_mask_to_5d
transformers.modeling_utils.PreTrainedModel.get_head_mask = get_head_mask
print("Transformers comprehensive compatibility bridge verified.")

from lavis.models import load_model_and_preprocess

# 1. Load Knowledge Base
print("Loading Wikipedia Knowledge Base into memory...")
t0 = time.time()
kb = WikipediaKnowledgeBase(KB_PATH)
kb_list = kb.load_knowledge_base()
kb_dict = {entry.url: entry for entry in kb_list}
del kb, kb_list
print(f"Knowledge Base loaded ({len(kb_dict)} articles in {time.time() - t0:.1f}s).")

# 2. Load Candidate Images Mapping
title2wikiimg = load_candidate_wiki_images(CANDIDATE_CSV, WIKI_IMG_PREFIX)

# 3. Load Model & Preprocessors
print(f"Loading Q-Former IT2IT Reranker model onto {device}...")
reranker_model, vis_processors, txt_processors = load_model_and_preprocess(
    name="qformer_IT2IT_reranker", model_type="pretrain", is_eval=True, device=device
)

print(f"Loading checkpoint: {RERANKER_CKPT}")
checkpoint = torch.load(RERANKER_CKPT, map_location=device)
msg = reranker_model.load_state_dict(checkpoint, strict=False)
print(f"Checkpoint loaded successfully. (Missing: {len(msg.missing_keys)}, Unexpected: {len(msg.unexpected_keys)})")

reranker_model = reranker_model.half()
preprocess = targetpad_transform(1.25, 224)


### Cell 5: Run Step 2 Multimodal Reranking Loop
Performs fine-grained token-level cross-attention matching and linear score blending.

In [ ]:
# Ensure get_head_mask is attached to PreTrainedModel
import transformers.modeling_utils
if not hasattr(transformers.modeling_utils.PreTrainedModel, "get_head_mask"):
    def _convert_head_mask_to_5d(self, head_mask, num_hidden_layers):
        if head_mask.dim() == 1:
            head_mask = head_mask.unsqueeze(0).unsqueeze(0).unsqueeze(-1).unsqueeze(-1)
            head_mask = head_mask.expand(num_hidden_layers, -1, -1, -1, -1)
        elif head_mask.dim() == 2:
            head_mask = head_mask.unsqueeze(1).unsqueeze(-1).unsqueeze(-1)
        dtype = getattr(self, "dtype", torch.float32)
        return head_mask.to(dtype=dtype)
    def get_head_mask(self, head_mask=None, num_hidden_layers=12, is_attention_chunked=False):
        if head_mask is not None:
            head_mask = self._convert_head_mask_to_5d(head_mask, num_hidden_layers)
            if is_attention_chunked:
                head_mask = head_mask.unsqueeze(-1)
        else:
            head_mask = [None] * num_hidden_layers
        return head_mask
    transformers.modeling_utils.PreTrainedModel._convert_head_mask_to_5d = _convert_head_mask_to_5d
    transformers.modeling_utils.PreTrainedModel.get_head_mask = get_head_mask

# Load Samples & Step 1 Results
sample_list, sample_header = load_csv_data(SAMPLE_FILE)
with open(STEP1_RESULT, "r", encoding="utf-8") as f:
    step1_result_dict = json.load(f)
print(f"Total Test Queries: {len(sample_list)}")
reranker_model.eval()
torch.cuda.empty_cache()

top_ks = [1, 5, 10, 20]
recalls = {k: 0 for k in top_ks}
retrieval_result = {}
step1_alpha = 0.9  # OMGM ACL 2025 default

t_start = time.time()

for it in tqdm(range(len(sample_list)), desc="Step 2 IT2IT Rerank"):
    example = get_test_question(it, sample_list, sample_header)
    ground_truth = example.get("wikipedia_url", "")
    data_id = example.get("data_id", f"E-VQA_{it}")

    # Retrieve Step 1 results for this query
    if data_id in step1_result_dict:
        step1_item = step1_result_dict[data_id]
    elif str(data_id) in step1_result_dict:
        step1_item = step1_result_dict[str(data_id)]
    elif f"E-VQA_{it}" in step1_result_dict:
        step1_item = step1_result_dict[f"E-VQA_{it}"]
    else:
        step1_item = list(step1_result_dict.values())[it]

    step1_sim = step1_item["retrieval_similarities"]
    step1_entities = step1_item["retrieved_entities"]

    # Load Query Image & Text
    q_img_id = example.get("dataset_image_ids", "").split("|")[0]
    q_img_path = resolve_query_image(q_img_id, QUERY_IMG_DIR)
    query_img_obj = PIL.Image.open(q_img_path).convert("RGB")
    ref_image = preprocess(query_img_obj).to(device).unsqueeze(0)
    question_text = txt_processors["eval"](example.get("question", ""))

    # Build Candidate (Image, Section) Pairs
    IT2IT_null = []
    candidate_images = []
    candidate_sections = []
    entry2ITindex = {}

    for entry_id, title_url in enumerate(step1_entities):
        wiki_title = title_url.get("title", "")
        wiki_url = title_url.get("url", "")

        if wiki_title not in title2wikiimg or len(title2wikiimg[wiki_title]) == 0:
            IT2IT_null.append(True)
            continue

        cand_img_path = title2wikiimg[wiki_title][0]["img_path"]
        if not os.path.exists(cand_img_path) or wiki_url not in kb_dict:
            IT2IT_null.append(True)
            continue

        entry = kb_dict[wiki_url]
        entry_sections = reconstruct_wiki_sections(entry)
        if len(entry_sections) == 0:
            IT2IT_null.append(True)
            continue

        try:
            cand_img_obj = PIL.Image.open(cand_img_path).convert("RGB")
        except Exception:
            IT2IT_null.append(True)
            continue

        candidate_images.extend([cand_img_obj] * len(entry_sections))
        processed_sections = [txt_processors["eval"](sec) for sec in entry_sections]
        candidate_sections.extend(processed_sections)
        entry2ITindex[entry_id] = [
            len(candidate_sections) - len(entry_sections) + i
            for i in range(len(entry_sections))
        ]
        IT2IT_null.append(False)

    # Compute Multimodal Cross-Attention
    if len(candidate_images) > 0:
        with torch.no_grad(), torch.amp.autocast("cuda"):
            query_fusion_embs = reranker_model.extract_features(
                {"image": ref_image, "text_input": question_text},
                mode="multimodal_query",
            )["multimodal_embeds"]

            rerank_step = 32
            candidate_embs_list = []
            for sp in range(0, len(candidate_images), rerank_step):
                batch_imgs = candidate_images[sp : sp + rerank_step]
                batch_secs = candidate_sections[sp : sp + rerank_step]
                proc_imgs = torch.stack(process_images_in_parallel(batch_imgs, preprocess)).to(device)
                split_embs = reranker_model.extract_features(
                    {"image": proc_imgs, "text_input": batch_secs},
                    mode="multimodal_candidate",
                )["multimodal_embeds"]
                candidate_embs_list.append(split_embs)

            candidate_fusion_embs = torch.cat(candidate_embs_list, dim=0)
            sim_q2c_token2token = torch.matmul(
                query_fusion_embs, candidate_fusion_embs.permute(0, 2, 1)
            ).squeeze()
            if sim_q2c_token2token.dim() == 2:
                sim_q2c_token2token = sim_q2c_token2token.unsqueeze(0)

            sim_q2c_querytoken, _ = sim_q2c_token2token.max(-1)
            query_token_length = sim_q2c_querytoken.size(-1)
            sim_q2c = sim_q2c_querytoken.sum(-1) / query_token_length

            step2_sim = [0.0] * len(step1_sim)
            for entry_id, is_null in enumerate(IT2IT_null):
                if not is_null and entry_id in entry2ITindex:
                    step2_sim[entry_id] = max([sim_q2c[idx].item() for idx in entry2ITindex[entry_id]])

            alpha_1 = step1_alpha
            alpha_2 = 1.0 - alpha_1
            scores = (
                alpha_1 * torch.tensor(step1_sim, device=device)
                + alpha_2 * torch.tensor(step2_sim, device=device)
            )
            sorted_scores, reranked_idx = torch.sort(scores, descending=True)
    else:
        sorted_scores = torch.tensor(step1_sim, device=device)
        reranked_idx = torch.arange(len(step1_sim), device=device)
        sim_q2c = []

    top_k_wiki = [step1_entities[i.item()]["url"] for i in reranked_idx]

    top_20_sec_sim = []
    for i in reranked_idx:
        idx = i.item()
        if idx in entry2ITindex and len(sim_q2c) > 0:
            top_20_sec_sim.append([sim_q2c[it_idx].item() for it_idx in entry2ITindex[idx]])
        else:
            top_20_sec_sim.append([])

    retrieval_result[data_id] = {
        "retrieved_entities": [step1_entities[i.item()] for i in reranked_idx],
        "retrieval_similarities": [s.item() for s in sorted_scores],
        "sec_sim": top_20_sec_sim
    }

    q_rec = eval_recall(top_k_wiki, ground_truth, top_ks)
    for k in top_ks:
        recalls[k] += q_rec[k]


### Cell 6: Evaluation Metrics & Save Output
Prints final Recall@1, 5, 10, 20 and exports `result.json` to `/kaggle/working/step2_results/`.

In [ ]:
total_q = len(sample_list)
elapsed = time.time() - t_start

print("=" * 60)
print("FINAL STEP 2 EVALUATION RESULTS (1,000 InfoSeek Test Queries)")
print(f"Total Time: {elapsed:.1f}s ({elapsed / total_q:.2f}s per query)")
print("=" * 60)
for k in top_ks:
    print(f"  Recall@{k:<2}: {recalls[k] / total_q:.4f} ({recalls[k]}/{total_q})")
print("=" * 60)

# Save result.json
out_path = os.path.join(OUTPUT_DIR, "result.json")
with open(out_path, "w", encoding="utf-8") as f:
    json.dump(retrieval_result, f, indent=4)

print(f"Successfully saved Step 2 results to: {out_path}")
print(f"File size: {os.path.getsize(out_path) / (1024 * 1024):.2f} MB")